In [ ]:
import time
import pandas as pd
import numpy as np
import yfinance as yf

# ============================================================
# 0) SYMBOL TRANSLATION
# ============================================================
def nyse_to_yahoo(symbol: str) -> str:
    s = symbol.strip()

    # class shares
    s = s.replace(".", "-")

    # preferred / series
    s = s.replace("$", "-")

    # units
    if s.endswith("-U"):
        s = s[:-2] + "-UN"

    # warrants
    if s.endswith("-W"):
        s = s[:-2] + "-WT"

    return s


def batched(seq, n):
    for i in range(0, len(seq), n):
        yield seq[i:i+n]


# ============================================================
# 1) GET CURRENT NYSE LIST
# ============================================================
url = "https://www.nasdaqtrader.com/dynamic/symdir/otherlisted.txt"

raw = pd.read_csv(url, sep="|")
raw = raw[raw["ACT Symbol"] != "File Creation Time"].copy()

nyse = raw.loc[
    (raw["Exchange"] == "N") &
    (raw["Test Issue"] == "N") &
    (raw["ETF"] == "N")
].copy()

nyse["symbol_nyse"] = nyse["ACT Symbol"].astype(str).str.strip()
nyse["symbol_yahoo"] = nyse["symbol_nyse"].map(nyse_to_yahoo)

symbols_yf = nyse["symbol_yahoo"].drop_duplicates().tolist()

print(f"NYSE symbols to request from Yahoo: {len(symbols_yf)}")


# ============================================================
# 2) ROBUST DOWNLOAD HELPERS
# ============================================================
START = "2010-01-01"
END = None

def download_batch(batch, start=START, end=END, pause=2):
    """
    Download a batch from Yahoo with conservative settings.
    """
    try:
        df = yf.download(
            tickers=batch,
            start=start,
            end=end,
            interval="1d",
            group_by="ticker",
            auto_adjust=False,
            threads=False,
            progress=False,
            actions=False,
            repair=False,
        )
        time.sleep(pause)
        return df
    except Exception as e:
        print(f"Batch failed for {batch[:3]}... ({len(batch)} tickers): {e}")
        time.sleep(pause)
        return pd.DataFrame()


def download_single(sym, start=START, end=END, pause=1.5):
    """
    Retry an individual symbol. Returns None if still unavailable.
    """
    try:
        df = yf.download(
            tickers=sym,
            start=start,
            end=end,
            interval="1d",
            auto_adjust=False,
            threads=False,
            progress=False,
            actions=False,
            repair=False,
        )
        time.sleep(pause)

        if df is None or df.empty:
            return None

        # For single-symbol download, columns are just Open/High/Low...
        cols = ["Open", "High", "Low", "Close", "Adj Close", "Volume"]
        existing = [c for c in cols if c in df.columns]
        if not existing:
            return None

        df.columns = pd.MultiIndex.from_product([[sym], existing])
        return df

    except Exception:
        time.sleep(pause)
        return None


def extract_downloaded_symbols(df):
    """
    Return symbols that actually came back with data.
    """
    if df is None or df.empty:
        return set()

    if isinstance(df.columns, pd.MultiIndex):
        return {c[0] for c in df.columns}
    return set()


# ============================================================
# 3) DOWNLOAD IN SMALL BATCHES + RETRY FAILURES
# ============================================================
all_frames = []
failed_candidates = []

batch_size = 50
pause_between_batches = 3

for i, batch in enumerate(batched(symbols_yf, batch_size), start=1):
    print(f"Downloading batch {i}...")
    df = download_batch(batch, pause=pause_between_batches)

    if df is not None and not df.empty:
        returned = extract_downloaded_symbols(df)
        missing = [sym for sym in batch if sym not in returned]

        if len(returned) > 0:
            all_frames.append(df)

        failed_candidates.extend(missing)
    else:
        failed_candidates.extend(batch)

print(f"Initial failed candidates: {len(failed_candidates)}")

# Deduplicate while preserving order
seen = set()
failed_candidates = [x for x in failed_candidates if not (x in seen or seen.add(x))]

# Retry one by one
recovered_frames = []
confirmed_bad = []

for j, sym in enumerate(failed_candidates, start=1):
    if j % 25 == 0:
        print(f"Retrying {j}/{len(failed_candidates)} ...")

    df1 = download_single(sym)

    if df1 is not None and not df1.empty:
        recovered_frames.append(df1)
    else:
        confirmed_bad.append(sym)

print(f"Recovered on retry: {len(recovered_frames)}")
print(f"Confirmed bad/unavailable: {len(confirmed_bad)}")

# Combine everything
if recovered_frames:
    all_frames.extend(recovered_frames)

if not all_frames:
    raise RuntimeError("No Yahoo Finance data downloaded.")

prices = pd.concat(all_frames, axis=1)
prices = prices.loc[:, ~prices.columns.duplicated()].sort_index(axis=1)

print(f"Downloaded symbols with data: {len({c[0] for c in prices.columns})}")


# ============================================================
# 4) SAVE EXCEPTION LIST
# ============================================================
exceptions = nyse.loc[
    nyse["symbol_yahoo"].isin(confirmed_bad),
    ["symbol_nyse", "symbol_yahoo", "Security Name", "Exchange", "ETF"]
].copy()

exceptions = exceptions.sort_values(["symbol_nyse"])
exceptions.to_csv("nyse_yahoo_confirmed_exceptions.csv", index=False)

print(f"Saved exceptions: {len(exceptions)}")


# ============================================================
# 5) EXTRACT HIGH / LOW PANELS
# ============================================================
tickers_in_prices = sorted({c[0] for c in prices.columns if isinstance(c, tuple)})

high = pd.DataFrame({
    ticker: prices[(ticker, "High")]
    for ticker in tickers_in_prices
    if (ticker, "High") in prices.columns
})

low = pd.DataFrame({
    ticker: prices[(ticker, "Low")]
    for ticker in tickers_in_prices
    if (ticker, "Low") in prices.columns
})

common = high.columns.intersection(low.columns)
high = high[common].sort_index()
low = low[common].sort_index()

# Save raw panels
high.to_parquet("nyse_high_panel.parquet")
low.to_parquet("nyse_low_panel.parquet")

print(f"Saved high/low panels with {len(common)} symbols.")


# ============================================================
# 6) COMPUTE DAILY NEW 52-WEEK HIGHS / LOWS
# ============================================================
lookback = 252

prior_252_high = high.shift(1).rolling(lookback, min_periods=lookback).max()
prior_252_low = low.shift(1).rolling(lookback, min_periods=lookback).min()

new_high = high.gt(prior_252_high)
new_low = low.lt(prior_252_low)

new_high_count = new_high.sum(axis=1)
new_low_count = new_low.sum(axis=1)
issues_traded = high.notna().sum(axis=1)

# avoid divide-by-zero
hlli_raw = np.where(
    issues_traded > 0,
    np.minimum(new_high_count, new_low_count) / issues_traded,
    np.nan
)
hlli_pct = 100 * pd.Series(hlli_raw, index=issues_traded.index)
hlli_10dma = hlli_pct.rolling(10).mean()

result = pd.DataFrame({
    "new_highs": new_high_count,
    "new_lows": new_low_count,
    "issues_traded": issues_traded,
    "hlli_pct": hlli_pct,
    "hlli_10dma": hlli_10dma,
}).dropna()

print(result.tail())

result.to_csv("nyse_high_low_logic_index.csv")
result.to_parquet("nyse_high_low_logic_index.parquet")

print("Saved HLLI series.")

NYSE symbols to request from Yahoo: 2815



48 Failed downloads:
['ABR-E', 'ACVA', 'ABX', 'ABXL', 'AEM', 'ACEL', 'ACP-A', 'AAMI', 'AB', 'ACH', 'AEG', 'ACM', 'A', 'AER', 'AEFC', 'ABM', 'ACHR-WT', 'ABG', 'ACR', 'ADC', 'ADM', 'AAUC', 'ADC-A', 'ACR-C', 'ABCB', 'ACP', 'ADT', 'ACN', 'ABEV', 'ACR-D', 'ABBV', 'ABT', 'ACRE', 'ACHR', 'ADX', 'AA', 'ACI', 'ADCT', 'ABR-F', 'ACV', 'ABR-D', 'AEE', 'ADNT', 'AD', 'AAP', 'AEO', 'ACA', 'ABR']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')



50 Failed downloads:
['AGM-H', 'AHT-F', 'AFGC', 'AHT-I', 'AES', 'AGO', 'AFGB', 'AGI', 'AHR', 'AIO', 'AHRT-A', 'AFGD', 'AHRT', 'AIT', 'AHT-D', 'AFGE', 'AI', 'AG', 'AIIA-R', 'AGM', 'AGRO', 'AGM-F', 'AGL', 'AGX', 'AIN', 'AEXA', 'AERO', 'AHL-E', 'AHT', 'AHT-H', 'AGCO', 'AESI', 'AGM-E', 'AFB', 'AIG', 'AHL-D', 'AGM-G', 'AHL-F', 'AGM-D', 'AIV', 'AGM-A', 'AHT-G', 'AIIA', 'AIR', 'AFG', 'AGBK', 'AIIA-UN', 'AGD', 'AII', 'AFL']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')



42 Failed downloads:
['ALL-H', 'AMC', 'ALLE', 'AMT', 'ALB', 'ALUB-WT', 'ALH', 'ALL-J', 'ALUB-UN', 'ALLY', 'AMH-H', 'AKO-B', 'ALUB', 'AIZN', 'ALK', 'AMN', 'ALL-I', 'AMBP', 'ALSN', 'AMPX', 'ALG', 'AMPY', 'AMH', 'AL', 'ALL-B', 'AMBQ', 'AKR', 'ALTG', 'AMPX-WT', 'AMG', 'ALB-A', 'AMR', 'AJG', 'AMRC', 'AKO-A', 'AMRZ', 'AM', 'ALTG-A', 'AMP', 'AMH-G', 'ALIT', 'AIZ']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')



44 Failed downloads:
['ARES-B', 'APAM', 'ARCO', 'ARE', 'ARLO', 'APOS', 'ARMK', 'ARW', 'ANET', 'APD', 'APH', 'ANF', 'ARIS', 'APO-A', 'ANVS', 'ARR-C', 'AMTD', 'ARDC', 'AMTM', 'AN', 'APLE', 'AMWL', 'AOMD', 'AP', 'APG', 'AQN', 'ARES', 'AOD', 'AROC', 'ARX', 'APTV', 'ARI', 'ANDG', 'AQNB', 'AOS', 'AR', 'ANRO', 'APO', 'AMTB', 'AOMR', 'ANGX', 'AS', 'AMX', 'ANG-D']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


...

$TRTN-A: possibly delisted; no timezone found

1 Failed download:
['TRTN-A']: possibly delisted; no timezone found
$TRTN-B: possibly delisted; no timezone found

1 Failed download:
['TRTN-B']: possibly delisted; no timezone found
$TRTN-C: possibly delisted; no timezone found

1 Failed download:
['TRTN-C']: possibly delisted; no timezone found


Retrying 275/300 ...


$TRTN-D: possibly delisted; no timezone found

1 Failed download:
['TRTN-D']: possibly delisted; no timezone found
$TRTN-E: possibly delisted; no timezone found

1 Failed download:
['TRTN-E']: possibly delisted; no timezone found
$TRTN-F: possibly delisted; no timezone found

1 Failed download:
['TRTN-F']: possibly delisted; no timezone found
$TRTN-G: possibly delisted; no timezone found

1 Failed download:
['TRTN-G']: possibly delisted; no timezone found
$TRTX-C: possibly delisted; no timezone found

1 Failed download:
['TRTX-C']: possibly delisted; no timezone found
$TSI-R: possibly delisted; no timezone found

1 Failed download:
['TSI-R']: possibly delisted; no timezone found

1 Failed download:
['TSLX']: TypeError("'NoneType' object is not subscriptable")


Retrying 300/300 ...
Recovered on retry: 247
Confirmed bad/unavailable: 53
Downloaded symbols with data: 2762
Saved exceptions: 53


ImportError: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - `Import pyarrow` failed. pyarrow is required for parquet support. Use pip or conda to install the pyarrow package.
 - `Import fastparquet` failed. fastparquet is required for parquet support. Use pip or conda to install the fastparquet package.

In [5]:
high.to_csv("nyse_high_panel.csv")
low.to_csv("nyse_low_panel.csv")

print(f"Saved high/low panels with {len(common)} symbols.")


Saved high/low panels with 2762 symbols.


In [4]:
lookback = 252

prior_252_high = high.shift(1).rolling(lookback, min_periods=lookback).max()
prior_252_low = low.shift(1).rolling(lookback, min_periods=lookback).min()

new_high = high.gt(prior_252_high)
new_low = low.lt(prior_252_low)

new_high_count = new_high.sum(axis=1)
new_low_count = new_low.sum(axis=1)
issues_traded = high.notna().sum(axis=1)

# avoid divide-by-zero
hlli_raw = np.where(
    issues_traded > 0,
    np.minimum(new_high_count, new_low_count) / issues_traded,
    np.nan
)
hlli_pct = 100 * pd.Series(hlli_raw, index=issues_traded.index)
hlli_10dma = hlli_pct.rolling(10).mean()

result = pd.DataFrame({
    "new_highs": new_high_count,
    "new_lows": new_low_count,
    "issues_traded": issues_traded,
    "hlli_pct": hlli_pct,
    "hlli_10dma": hlli_10dma,
}).dropna()

print(result.tail())

result.to_csv("nyse_high_low_logic_index.csv")
# result.to_parquet("nyse_high_low_logic_index.parquet")

print("Saved HLLI series.")

            new_highs  new_lows  issues_traded  hlli_pct  hlli_10dma
Date                                                                
2026-03-06          7        13            683  1.024890    1.966320
2026-03-09         12        26            682  1.759531    1.774626
2026-03-10          4        10            684  0.584795    1.421945
2026-03-11          5        16            683  0.732064    1.304256
2026-03-12          9        27            680  1.323529    1.319307
Saved HLLI series.


In [6]:
!pip install pandas


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## CSV Time-Series Analysis

This section reads `nyse_high_low_logic_index.csv`, plots the daily HLLI against its 10-day average, and prints a compact regime summary.

Note: the export is pinned at `0.0` through 2018 and the first non-zero observation is `2019-01-03`, so treat 2018 as a warm-up / construction artifact rather than a calm-market regime.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.ticker import PercentFormatter

csv_path = Path('nyse_high_low_logic_index.csv')
df = pd.read_csv(csv_path, parse_dates=['Date']).sort_values('Date').reset_index(drop=True)

first_non_zero = df.loc[df['hlli_pct'].gt(0), 'Date'].min()
max_hlli = df['hlli_pct'].max()

fig, axes = plt.subplots(
    2,
    1,
    figsize=(15, 9),
    sharex=True,
    gridspec_kw={'height_ratios': [2.2, 1]},
)

axes[0].plot(df['Date'], df['hlli_pct'], color='#1d4e89', linewidth=1.15, alpha=0.85, label='Daily HLLI')
axes[0].plot(df['Date'], df['hlli_10dma'], color='#d1495b', linewidth=2.0, label='10-day average')
axes[0].axhspan(2, 3, color='#f4a261', alpha=0.15)
axes[0].axhspan(3, max_hlli + 0.3, color='#e76f51', alpha=0.12)
axes[0].axhline(2, color='#f4a261', linestyle='--', linewidth=1)
axes[0].axhline(3, color='#e76f51', linestyle='--', linewidth=1)
axes[0].annotate(
    f'First non-zero: {first_non_zero:%Y-%m-%d}',
    xy=(first_non_zero, 0),
    xytext=(12, 18),
    textcoords='offset points',
    fontsize=9,
    color='#444444',
    arrowprops={'arrowstyle': '-', 'color': '#777777', 'lw': 0.8},
)
axes[0].set_title('NYSE High-Low Logic Index', fontsize=15, pad=12)
axes[0].set_ylabel('Percent of issues')
axes[0].yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))
axes[0].grid(axis='y', alpha=0.25)
axes[0].legend(frameon=False, ncol=2, loc='upper right')

axes[1].plot(df['Date'], df['new_highs'], color='#2a9d8f', linewidth=1.15, label='New highs')
axes[1].plot(df['Date'], df['new_lows'], color='#6d597a', linewidth=1.15, label='New lows')
axes[1].set_ylabel('Count')
axes[1].set_xlabel('Date')
axes[1].grid(axis='y', alpha=0.25)
axes[1].legend(frameon=False, ncol=2, loc='upper right')

plt.tight_layout()
fig.savefig('nyse_high_low_logic_index_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
from IPython.display import display

sample_mean = df['hlli_pct'].mean()
sample_median = df['hlli_pct'].median()
recent_20_mean = df['hlli_pct'].tail(20).mean()
recent_60_mean = df['hlli_pct'].tail(60).mean()
share_gt_2 = df['hlli_pct'].gt(2).mean() * 100
share_gt_3 = df['hlli_pct'].gt(3).mean() * 100
latest = df.iloc[-1]
latest_pctile = df['hlli_pct'].rank(pct=True).iloc[-1] * 100
latest_dma_pctile = df['hlli_10dma'].rank(pct=True).iloc[-1] * 100


def longest_streak(mask: pd.Series):
    groups = mask.ne(mask.shift()).cumsum()
    streak_lengths = mask.groupby(groups).cumsum()
    if streak_lengths.max() == 0:
        return 0, None
    end_idx = streak_lengths.idxmax()
    return int(streak_lengths.max()), df.loc[end_idx, 'Date'].date()


regime = 'contained'
if latest['hlli_10dma'] > 3:
    regime = 'stress regime'
elif latest['hlli_10dma'] > 2:
    regime = 'elevated dispersion'
elif latest['hlli_10dma'] > 1:
    regime = 'mildly elevated'

streak_1, streak_1_end = longest_streak(df['hlli_pct'].gt(1))
streak_2, streak_2_end = longest_streak(df['hlli_pct'].gt(2))
streak_3, streak_3_end = longest_streak(df['hlli_pct'].gt(3))

top_days = df.nlargest(5, 'hlli_pct')[['Date', 'hlli_pct', 'new_highs', 'new_lows', 'issues_traded']].copy()
top_days['Date'] = top_days['Date'].dt.date
top_days['hlli_pct'] = top_days['hlli_pct'].round(3)

top_dma = df.nlargest(5, 'hlli_10dma')[['Date', 'hlli_10dma']].copy()
top_dma['Date'] = top_dma['Date'].dt.date
top_dma['hlli_10dma'] = top_dma['hlli_10dma'].round(3)

yearly = (
    df.assign(year=df['Date'].dt.year)
      .groupby('year')
      .agg(
          trading_days=('Date', 'size'),
          avg_hlli_pct=('hlli_pct', 'mean'),
          median_hlli_pct=('hlli_pct', 'median'),
          max_hlli_pct=('hlli_pct', 'max'),
          avg_issues=('issues_traded', 'mean'),
      )
)

yearly_peaks = (
    df.loc[df.groupby(df['Date'].dt.year)['hlli_pct'].idxmax(), ['Date', 'hlli_pct']]
      .assign(year=lambda frame: frame['Date'].dt.year, peak_date=lambda frame: frame['Date'].dt.date)
      .set_index('year')[['peak_date']]
)

yearly = yearly.join(yearly_peaks)
yearly[['avg_hlli_pct', 'median_hlli_pct', 'max_hlli_pct', 'avg_issues']] = yearly[
    ['avg_hlli_pct', 'median_hlli_pct', 'max_hlli_pct', 'avg_issues']
].round({'avg_hlli_pct': 3, 'median_hlli_pct': 3, 'max_hlli_pct': 3, 'avg_issues': 1})

print(f"Sample: {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d} ({len(df):,} trading days)")
print(
    f"Latest: {latest['Date']:%Y-%m-%d} | HLLI {latest['hlli_pct']:.2f}% | 10DMA {latest['hlli_10dma']:.2f}% | "
    f"new highs {int(latest['new_highs'])} | new lows {int(latest['new_lows'])} | issues traded {int(latest['issues_traded'])}"
)
print(
    f"Context: full-sample mean {sample_mean:.2f}% vs median {sample_median:.2f}% | "
    f"20-day mean {recent_20_mean:.2f}% | 60-day mean {recent_60_mean:.2f}%"
)
print(
    f"Current regime: {regime} | latest daily reading is at the {latest_pctile:.1f}th percentile | "
    f"10DMA is at the {latest_dma_pctile:.1f}th percentile"
)
print(f"Frequency: {share_gt_2:.2f}% of days closed above 2% and {share_gt_3:.2f}% above 3%")
print(f"Longest streaks: >1% = {streak_1} days ending {streak_1_end}, >2% = {streak_2} days ending {streak_2_end}, >3% = {streak_3} days ending {streak_3_end}")
print()
print('Top daily HLLI spikes:')
display(top_days)
print('Top 10-day average peaks:')
display(top_dma)
print('Yearly regime summary:')
display(yearly)
